In [ ]:
# !pip install matplotlib
# !pip install seaborn
# !pip install openai
# !pip install scipy
# !pip install z3-solver
# !pip install dotenv

import sys
sys.path.append('../../utils')
import utility
import numpy as np
import os
import pickle
import json
import re
from z3 import *

seeds_considered = [3,12,26,85,107]

In [ ]:
models_comparison = ['gpt-4o-mini', 'o3-mini', 'Qwen3-8B_think', 'Qwen3-30B-A3B_think']
clean_outputs_FOLIO = {}
clean_outputs_Stanford = {}

for model in models_comparison:
    clean_outputs_FOLIO[model] = utility.load_pickle(f'../Logical Translation/Results/Clean Results/FOLIO_log_transl_{model}_seed.pkl')
    clean_outputs_Stanford[model] = utility.load_pickle(f'../Logical Translation/Results/Clean Results/Stanford_log_transl_{model}_seed.pkl')

print(clean_outputs_Stanford['gpt-4o-mini'][3])


FOLIO


In [ ]:
folio_dict = utility.make_jsonl_into_list_dictionaries('../../datasets/folio-train-clean.jsonl')
NL_sentences = {}
FOL_formulas = {}
story_id_already_seen = set()
for item in folio_dict:
    story_id = item['story_id']
    if story_id not in story_id_already_seen:
        story_id_already_seen.add(story_id)
        NL_sentences[story_id] = item['premises']
        FOL_formulas[story_id] = item['premises-FOL']
# remove the story 236 because logically problematic (PValue doesn't mantain the same arity in the FOL translations)
NL_sentences.pop(236)
FOL_formulas.pop(236)


        
# Delete the sentences in the stories that have the Xor symbol ⊕ (it is treated inconsistently in the dataset)
for i in range(2):
    for story_id in FOL_formulas.keys():
        for sentence in FOL_formulas[story_id]:
            if '⊕' in sentence:
                index = FOL_formulas[story_id].index(sentence)
                FOL_formulas[story_id].pop(index)
                NL_sentences[story_id].pop(index)


# Use the same pipeline used in Logical_translation_FOLIO.ipynb to get the true/false vector
vector_translations_FOLIO = {}

for model in models_comparison:
    predictions = {}
    FOL_formulas_for_Z3 = {}
    predictions_for_Z3 = {}
    list_exception = {}
    list_exception_use_equality = {}

    for seed in seeds_considered:
        predictions[seed] = {}
        FOL_formulas_for_Z3[seed] = {}
        predictions_for_Z3[seed] = {}
        list_exception[seed] = []
        list_exception_use_equality[seed] = []


    for seed in seeds_considered:
        for story_id in NL_sentences.keys():
            predictions[seed][story_id] = ['' for i in NL_sentences[story_id]]
            FOL_formulas_for_Z3[seed][story_id] = ['' for i in NL_sentences[story_id]]
            predictions_for_Z3[seed][story_id] = ['' for i in NL_sentences[story_id]]
        for line in clean_outputs_FOLIO[model][seed]:
            story_id, number_instance, answer = line['story_id'], line['number_instance'], line['answer']
            predictions[seed][story_id][number_instance] = answer


        for story_id in FOL_formulas.keys():
            for i, formula in enumerate(FOL_formulas[story_id]):
                try:
                
                    x = formula
                    y = predictions[seed][story_id][i]
                    
                    # - put in a nicer form to be parsed:
                    # replace [ or ] with ( or )
                    y = y.replace('[', '(')
                    y = y.replace(']', ')')
                    # - put a space after the quantifiers and remove the space between the quantifier and the varibale
                    y = y.replace('∀ ', '∀')
                    y = y.replace('∃ ', '∃')
                    # - add a space after the quantified variable
                    pattern = r"∀([a-zA-Z]+)\("
                    replacement = r"∀\1 ("
 
                    y = re.sub(pattern, replacement, y)
                    
                    pattern = r"∃([a-zA-Z]+)\("
                    replacement = r"∃\1 ("
                    y = re.sub(pattern, replacement, y)
                    # remove final dots or comma for ending 
                    if y[-1] == ',' or y[-1] == '.':
                        y = y[:-1]
                    
                    # Order the constant, and predicate symbols from the longer to the shorter (in this way we don't have problem if we have two constants like 'son' and 'jamson')
                    
                    r, c1, v1 = utility.extract_rel_const_var(x)
                    r2, c2, v2 = utility.extract_rel_const_var(y)
                    c = c1.union(c2)
                    v = v1.union(v2)
                    r.update(r2)

                    
                    list_constants = sorted(list(c), key=len, reverse=True)

                    unary_symbols = [rel for rel in r.keys() if r[rel] == 1]
                    binary_symbols = [rel for rel in r.keys() if r[rel] == 2]
                    ternary_symbols = [rel for rel in r.keys() if r[rel] == 3]
                    
                    
                    list_replacements = sorted(list_constants + unary_symbols + binary_symbols + ternary_symbols, key=len, reverse=True)
                
                    for item in list_replacements:
                        if item in list_constants:
                            index = list_constants.index(item) + 1
                            x = x.replace(item, 'A' + str(index))
                            y = y.replace(item, 'A' + str(index))
                        if item in unary_symbols:
                            index = unary_symbols.index(item) + 1
                            x = x.replace(item, 'U'+str(index))
                            y = y.replace(item, 'U'+str(index))
                        if item in binary_symbols:
                            index = binary_symbols.index(item) + 1
                            x = x.replace(item, 'B'+str(index)) 
                            y = y.replace(item, 'B'+str(index))
                        if item in ternary_symbols:
                            index = ternary_symbols.index(item) + 1
                            x = x.replace(item, 'T'+str(index))
                            y = y.replace(item, 'T'+str(index))
                    

                    FOL_formulas_for_Z3[seed][story_id][i] = utility.parse_formula_SMTLIB_universal(x)
                    predictions_for_Z3[seed][story_id][i] = utility.parse_formula_SMTLIB_universal(y)
                    
                except:
                    list_exception[seed].append(f'{story_id}_{i}')
                    if ('=' in predictions[seed][story_id][i]) or ('≠' in predictions[seed][story_id][i]):
                        list_exception_use_equality[seed].append(f'{story_id}_{i}')

    correct_translations = {}

    for seed in seeds_considered:
        correct_translations[seed] = {}

    for seed in seeds_considered:
        print(len(list_exception[seed]))
        for story_id in NL_sentences.keys():
            correct_translations[seed][story_id] = ['' for i in NL_sentences[story_id]]
            for i, sent in enumerate(NL_sentences[story_id]):
                key = f'{story_id}_{i}'
               
                if key in list_exception[seed]:
                    if key in list_exception_use_equality[seed]:
                        correct_translations[seed][story_id][i] = False
                    else:
                        correct_translations[seed][story_id][i] = False
                else:
                    pred = predictions_for_Z3[seed][story_id][i]
                    correct_translations[seed][story_id][i] = utility.check_equivalence_with_parsed_formulas(pred, FOL_formulas_for_Z3[seed][story_id][i])
                
    
    # Save in the dictionary vector_translation
    vector_translations_FOLIO[model] = correct_translations

In [ ]:
ok_ko = {}
model1 = 'gpt-4o-mini'
model2 = 'o3-mini'
models_to_compare = [('gpt-4o-mini', 'o3-mini'),
                     ('Qwen3-8B_think', 'Qwen3-30B-A3B_think'),
                     ('gpt-4o-mini', 'Qwen3-8B_think')]

for x in models_to_compare:
    model1, model2 = x
    print(model1, model2)
    for seed in seeds_considered:
        ok_ko[seed] = [0, 0, 0, 0] 
        # mean ok_ok, ok_ko, ko_ok, ko_ko
        for story_id in FOL_formulas.keys():
            mod1 = vector_translations_FOLIO[model1][seed][story_id]
            mod2 = vector_translations_FOLIO[model2][seed][story_id]
            assert len(mod1) == len(mod2)
            for i in range(len(mod1)):
                entry1, entry2 = mod1[i], mod2[i]
                if entry1 and entry2:
                    ok_ko[seed][0] += 1
                elif entry1 and not entry2:
                    ok_ko[seed][1] += 1
                elif not entry1 and entry2:
                    ok_ko[seed][2] += 1
                elif not entry1 and not entry2:
                    ok_ko[seed][3] += 1
                else:
                    print('Fail')
        assert (sum(ok_ko[seed]) == 1565)
    avg = [0, 0, 0, 0]
    for i in range(len(avg)):
        for seed in seeds_considered:
            avg[i] += ok_ko[seed][i]
    avg = [x/(1565*len(seeds_considered)) for x in avg]

    print('Average', avg)


Stanford

In [ ]:
FOL_sentences = [x for x in utility.make_txt_into_list('../../datasets/FOLsentence.txt')]
number_instances = len(FOL_sentences)
ground_truths = utility.make_txt_into_list('../../datasets/ground-truth.txt')

number_instances = len(FOL_sentences)
seeds_considered = [3, 12, 107, 85, 26]



vector_translations_Stanford = {}

for model in models_comparison:
    predictions = {}
    FOL_formulas_for_Z3 = {}
    predictions_for_Z3 = {}
    list_exception = {}
    list_exception_use_equality = {}

    for seed in seeds_considered:
        predictions[seed] = ['' for i in FOL_sentences]
        FOL_formulas_for_Z3[seed] = ['' for i in FOL_sentences]
        predictions_for_Z3[seed] = ['' for i in FOL_sentences]
        list_exception[seed] = []
        list_exception_use_equality[seed] = []

    for seed in seeds_considered:
        for line in clean_outputs_Stanford[model][seed]:
            number_instance, answer = line['number_instance'], line['answer']
            predictions[seed][number_instance] = answer
            
        for i, formula in enumerate(FOL_sentences):
            try:
                x = formula
                y = predictions[seed][i]
                
                mask = (seed==3) and (i==149)
                
                # - put in a nicer form to be parsed:
                # replace [ or ] with ( or )
                y = y.replace('[', '(')
                y = y.replace(']', ')')
                # - put a space after the quantifiers and remove the space between the quantifier and the varibale
                y = y.replace('∀ ', '∀')
                y = y.replace('∃ ', '∃')
                # - add a space after the quantified variable
                pattern = r"∀([a-zA-Z]+)\("
                replacement = r"∀\1 ("
                y = re.sub(pattern, replacement, y)

                pattern = r"∃([a-zA-Z]+)\("
                replacement = r"∃\1 ("
                y = re.sub(pattern, replacement, y)
                # remove final dots or comma for ending 
                if y[-1] == ',' or y[-1] == '.':
                    y = y[:-1]

                # Order the constant, and predicate symbols from the longer to the shorter (in this way we don't have problem if we have two constants like 'son' and 'jamson')
                r, c1, v1 = utility.extract_rel_const_var(x)
                r2, c2, v2 = utility.extract_rel_const_var(y)
                c = c1.union(c2)
                v = v1.union(v2)
                r.update(r2)

                list_constants = sorted(list(c), key=len, reverse=True)

                unary_symbols = [rel for rel in r.keys() if r[rel] == 1]
                binary_symbols = [rel for rel in r.keys() if r[rel] == 2]
                ternary_symbols = [rel for rel in r.keys() if r[rel] == 3]
                
                list_replacements = sorted(list_constants + unary_symbols + binary_symbols + ternary_symbols, key=len, reverse=True)


                # Build a lookup dict of replacements
                replacements = {}
                for item in list_replacements:
                    if item in list_constants:
                        index = list_constants.index(item) + 1
                        replacements[item] = 'A' + str(index)
                    elif item in unary_symbols:
                        index = unary_symbols.index(item) + 1
                        replacements[item] = 'U' + str(index)
                    elif item in binary_symbols:
                        index = binary_symbols.index(item) + 1
                        replacements[item] = 'B' + str(index)
                    elif item in ternary_symbols:
                        index = ternary_symbols.index(item) + 1
                        replacements[item] = 'T' + str(index)

                # Compile one regex that matches any key
                pattern = re.compile('|'.join(map(re.escape, replacements.keys())))

                # Replace using the mapping
                def replace_all(match):
                    return replacements[match.group(0)]

                x_new = pattern.sub(replace_all, x)
                y_new = pattern.sub(replace_all, y)

                FOL_formulas_for_Z3[seed][i] = utility.parse_formula_SMTLIB_universal(x_new)
                predictions_for_Z3[seed][i] = utility.parse_formula_SMTLIB_universal(y_new)
                
                    
            except:
                list_exception[seed].append(f'{i}')
                if ('=' in predictions[seed][i]) or ('≠' in predictions[seed][i]):
                    list_exception_use_equality[seed].append(f'{i}')
    correct_translations = {}


    for seed in seeds_considered:
        correct_translations[seed] = [False for i in FOL_sentences]

    for seed in seeds_considered:
        for i, sent in enumerate(FOL_sentences):
            key = f'{i}'
            if key in list_exception[seed]:
                if key in list_exception_use_equality[seed]:
                    correct_translations[seed][i] = False
                else:
                    correct_translations[seed][i] = False
            else:
                pred = predictions_for_Z3[seed][i]
                correct_translations[seed][i] = utility.check_equivalence_with_parsed_formulas(pred, FOL_formulas_for_Z3[seed][i])
        
        
    # Save in the dictionary vector_translation
    vector_translations_Stanford[model] = correct_translations




In [ ]:
ok_ko = {}
model1 = 'gpt-4o-mini'
model2 = 'o3-mini'
models_to_compare = [('gpt-4o-mini', 'o3-mini'),
                     ('Qwen3-8B_think', 'Qwen3-30B-A3B_think'),
                     ('gpt-4o-mini', 'Qwen3-8B_think')]

for x in models_to_compare:
    model1, model2 = x
    print(model1, model2)
    for seed in seeds_considered:
        ok_ko[seed] = [0, 0, 0, 0] 
        # mean ok_ok, ok_ko, ko_ok, ko_ko
        for i in range(len(FOL_sentences)):
            entry1 = vector_translations_Stanford[model1][seed][i]
            entry2 = vector_translations_Stanford[model2][seed][i]

            if entry1 and entry2:
                ok_ko[seed][0] += 1
            elif entry1 and not entry2:
                ok_ko[seed][1] += 1
            elif not entry1 and entry2:
                ok_ko[seed][2] += 1
            elif not entry1 and not entry2:
                ok_ko[seed][3] += 1
            else:
                print('Fail')
        assert (sum(ok_ko[seed]) == 159)
        percentage = [i/sum(ok_ko[seed]) for i in ok_ko[seed]]
        
    avg = [0, 0, 0, 0]
    for i in range(len(avg)):
        for seed in seeds_considered:
            avg[i] += ok_ko[seed][i]
    avg = [x/(159*len(seeds_considered)) for x in avg]

    print('Average', avg)
